In [0]:
from pyspark.sql.functions import expr, col, coalesce, lit, monotonically_increasing_id, when

In [0]:
# ── PART 1: Load the raw table and check its size
df_init = spark.read.table("group3_gp.testing.high_volume_fhv")
print(f"Initial row count: {df_init.count():,}")
print(f"Initial column count: {len(df_init.columns)}")
display(df_init.columns)

In [0]:
# Remove unuseful columns
cols_to_drop = {
    "sr_flag",
    "originating_base_num",
    "dispatching_base_num",
    "congestion_surcharge",
    "wav_request_flag",
    "wav_match_flag",
    "cbd_congestion_fee",
    "shared_request_flag",
    "access_a_ride_flag"   
}

df_clean = df_init.drop(*cols_to_drop)
print(f"Column count after drop: {len(df_clean.columns)}")



In [0]:
from pyspark.sql.functions import expr, lit
# Convert every column from raw strings to proper types using try_cast for safety.

column_types = {
    "dolocationid": "integer",
    "dropoff_datetime": "timestamp",
    "hvfhs_license_num": "string",
    "pickup_datetime": "timestamp",
    "pulocationid": "integer",
    "Year": "integer",
    "base_passenger_fare": "decimal(10,2)",
    "bcf": "decimal(10,2)",
    "driver_pay": "decimal(10,2)",
    "on_scene_datetime": "timestamp",
    "originating_base_num": "string",
    "request_datetime": "timestamp",
    "sales_tax": "decimal(10,2)",
    "shared_match_flag": "string",
    "tips": "decimal(10,2)",
    "tolls": "decimal(10,2)",
    "trip_miles": "double",
    "trip_time": "double",
}

# Cast all columns to their proper types
for col, dtype in column_types.items():
    if col in df_clean.columns:
        df = df_clean.withColumn(col, expr(f"try_cast({col} as {dtype})"))

# Tag every row as "High Volume FHV"

df = df.withColumn("taxi_type", lit("High Volume FHV"))
display(df.columns)

### **AVBRÖT CLEANING HÄR... SES SEN **

In [0]:
column_types = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp"
}

df = df_clean

for c, dtype in column_types.items():
    if c in df.columns:
        df = df.withColumn(c, expr(f"try_cast({c} as {dtype})"))

df = (
    df
    .withColumn("service_type", lit("High Volume FHV"))
    .withColumn("taxi_type", lit("unknown"))
    .withColumn("pulocationid", col("pulocationid").cast("double").cast("int"))
    .withColumn("dolocationid", col("dolocationid").cast("double").cast("int"))
)

print(f"Initial row count: {df.count():,}")

In [0]:
# Map raw payment type codes (still strings) to descriptions.
# Both legacy string codes (CRD, CSH, etc.) and numeric codes ("1", "2", etc.)
# are handled here BEFORE the type cast converts them to integers.
from pyspark.sql import functions as F

In [0]:
# Drop rows that are clearly invalid:
# missing timestamps, dropoff before pickup, zero/negative fares,
# passenger count outside 1-8.

final_df = df.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    (
        F.col("pulocationid").isNotNull() & 
        F.col("dolocationid").isNotNull()
    )
)
print(f"Rows after filtering: {final_df.count()}")

In [0]:
final_df.createOrReplaceTempView("final_df")

In [0]:
%sql
SELECT dolocationid, COUNT(*) FROM final_df WHERE dolocationid > 263 OR dolocationid < 1 GROUP BY dolocationid

In [0]:
%sql
SELECT
  year,
  COUNT(*) AS total
FROM final_df
GROUP BY year
ORDER BY year;